# DIMER Language-Model Fine-Tuning — Standalone Colab

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/language-model-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/language-model-pipeline/blob/main/tutorials/language_model_finetuning_colab.ipynb)

This notebook is an educational, standalone version of the DIMER language-model supervised fine-tuning capability.

> approved pinned base → Hugging Face or DIMER ZIP → sample/BYOD → validation → baseline generation → QLoRA SFT → before/after generation → new prompts → adapter export → fresh reload

## What you will learn

You will see how a **base model**, **tokenizer**, and **PEFT adapter** fit together; why immutable revisions matter; how a chat template turns conversations into tokens; why we apply **assistant-only** loss masking; what QLoRA changes compared with full training; how to interpret optimization metrics without confusing them with task quality; and why a clean reload is part of the artifact contract.

We keep three kinds of evidence separate. **Pipeline evidence** says the workflow executed. **Optimization evidence** includes loss, perplexity, runtime, supervised token counts, and GPU memory. **Task quality** asks whether the adapted model is actually better for its intended use. The few prompt comparisons in this tutorial are illustrative, not a benchmark.

**Default:** SmolLM3 3B. **AI provenance:** OpenAI / ChatGPT — GPT-5.6 Sol High, Builder role; not independent reviewer sign-off.


## 1. Runtime

Fine-tuning a multi-billion-parameter language model is hardware-sensitive. QLoRA reduces memory by keeping the frozen base weights in 4-bit form, but activations, gradients, adapter parameters, optimizer state, and temporary tensors still consume VRAM. We therefore require a CUDA runtime and print the assigned GPU and memory.

The package versions are pinned because tokenizer templates, PEFT behavior, quantization support, and model-loading semantics can change across releases. This does not guarantee bit-for-bit determinism, but it makes the run substantially easier to reproduce.


In [ ]:
%pip -q install transformers==4.57.1 tokenizers==0.22.1 huggingface-hub==0.36.0 peft==0.18.0 accelerate==1.11.0 bitsandbytes==0.49.0 safetensors==0.8.0 "datasets>=3,<5"


In [ ]:
import gc,hashlib,json,math,os,platform,random,re,shutil,stat,time,zipfile
from pathlib import Path,PurePosixPath
import pandas as pd,torch
from datasets import load_dataset
from huggingface_hub import HfApi
from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig
from peft import LoraConfig,PeftModel,get_peft_model,prepare_model_for_kbit_training
if not torch.cuda.is_available(): raise RuntimeError("Use a Colab GPU runtime")
GPU_NAME=torch.cuda.get_device_name(0); GPU_VRAM_GB=torch.cuda.get_device_properties(0).total_memory/1024**3
AI_PROVENANCE={"provider":"OpenAI","product":"ChatGPT","model":"GPT-5.6 Sol High","role":"Builder","note":"Not independent reviewer sign-off"}
print(platform.python_version(),torch.__version__,GPU_NAME,f"{GPU_VRAM_GB:.2f} GiB")


## 2. Choose a base model

The notebook does not accept arbitrary Hugging Face IDs. Every option is pinned to a known model ID and immutable commit SHA, and `trust_remote_code=False` is fixed.

SmolLM3 3B is the default tutorial candidate. Its production VRAM minimum remains unpublished until measured. Qwen3 1.7B, Qwen3 4B, and Granite 4.1 3B keep the measured QLoRA thresholds already recorded in the project.

For **Llama 3.2 3B Instruct**, the repository is gated. This is useful because Colab has a Secrets panel: add a secret named `HF_TOKEN`, enable **Notebook access**, and make sure the same Hugging Face account has accepted the Llama terms. A successful run proves the Colab credential path works for the pinned checkpoint. It does **not** prove that DIMER itself can inject a Hugging Face credential, and it does not automatically grant redistribution rights for DIMER-hosted weights.


In [ ]:
TUTORIAL_REGISTRY={
"smollm3-3b":{"model_id":"HuggingFaceTB/SmolLM3-3B","revision":"a07cc9a04f16550a088caea529712d1d335b0ac1","license":"apache-2.0","min_vram_gb":None,"requires_hf_token":False,"dimer_zip":True,"state":"tutorial-candidate/internal-only"},
"qwen3-1.7b":{"model_id":"Qwen/Qwen3-1.7B","revision":"70d244cc86ccca08cf5af4e1e306ecf908b1ad5e","license":"apache-2.0","min_vram_gb":9.3,"requires_hf_token":False,"dimer_zip":True,"state":"user-facing"},
"qwen3-4b":{"model_id":"Qwen/Qwen3-4B","revision":"1cfa9a7208912126459214e8b04321603b3df60c","license":"apache-2.0","min_vram_gb":11.5,"requires_hf_token":False,"dimer_zip":True,"state":"user-facing"},
"granite-4.1-3b":{"model_id":"ibm-granite/granite-4.1-3b","revision":"c0650403e44e78ec0262dab1c90914c65b196c4e","license":"apache-2.0","min_vram_gb":8.7,"requires_hf_token":False,"dimer_zip":True,"state":"user-facing"},
"llama-3.2-3b-instruct":{"model_id":"meta-llama/Llama-3.2-3B-Instruct","revision":"0cb88a4f764b7a12671c53f0838cd831a0843b95","license":"llama3.2","min_vram_gb":None,"requires_hf_token":True,"dimer_zip":False,"state":"credential-test-candidate"}}
BASE_MODEL_KEY = "smollm3-3b" # @param ["smollm3-3b","qwen3-1.7b","qwen3-4b","granite-4.1-3b","llama-3.2-3b-instruct"]
MODEL_SOURCE = "Pinned Hugging Face" # @param ["Pinned Hugging Face","DIMER ZIP"]
TRAINING_METHOD = "qlora" # @param ["qlora"]
MAX_SEQUENCE_LENGTH = 512 # @param {type:"integer"}
EPOCHS = 1 # @param {type:"integer"}
LEARNING_RATE = 0.0002 # @param {type:"number"}
LORA_RANK = 8 # @param {type:"integer"}
SEED = 42 # @param {type:"integer"}
entry=TUTORIAL_REGISTRY[BASE_MODEL_KEY]; model_id=entry["model_id"]; revision=entry["revision"]; base_license=entry["license"]
if MODEL_SOURCE=="DIMER ZIP" and not entry["dimer_zip"]: raise RuntimeError("This model is not approved for the DIMER ZIP path; use Pinned Hugging Face.")
if entry["min_vram_gb"] is None: print("⚠ Unmeasured tutorial candidate: this run records observed VRAM; it does not set a production minimum.")
elif GPU_VRAM_GB<entry["min_vram_gb"]: raise RuntimeError(f"Need at least {entry['min_vram_gb']} GiB for the measured profile")
print(BASE_MODEL_KEY,model_id,revision,entry["state"])


## 3. Hugging Face credentials and model acquisition

Public checkpoints need no token. For a gated checkpoint, we read `HF_TOKEN` only when required and perform a small metadata request before downloading model weights. This gives a fast, explicit credential test. The token is never printed and is never serialized into the adapter artifact.

`Pinned Hugging Face` loads the exact upstream revision. `DIMER ZIP` is for a DIMER-hosted copy of an approved model. A DIMER ZIP is not trusted merely because of its source: it must contain one `dimer-base-manifest.json` whose `format` is `dimer_hf_snapshot`, whose model identity matches the selected registry entry, and whose file sizes and SHA-256 hashes match the extracted snapshot. We reject path traversal and symlinks.

Llama deliberately has `dimer_zip=False`: credentials answer **access**, not whether DIMER may redistribute the weights.


In [ ]:
HF_TOKEN=None
if entry["requires_hf_token"]:
 from google.colab import userdata
 try: HF_TOKEN=userdata.get("HF_TOKEN")
 except Exception as exc: raise RuntimeError("Add HF_TOKEN in Colab Secrets, enable Notebook access, and accept the model terms.") from exc
 info=HfApi(token=HF_TOKEN).model_info(model_id,revision=revision)
 if info.sha!=revision: raise RuntimeError("Pinned Hugging Face revision mismatch")
 print("✓ Hugging Face credential/revision preflight passed.")

UPLOAD_DIMER_ZIP = False # @param {type:"boolean"}
DIMER_ZIP_PATH = "/content/dimer-base-model.zip" # @param {type:"string"}
EXPECTED_DIMER_ZIP_SHA256 = "" # @param {type:"string"}

def sha256_file(p):
 h=hashlib.sha256()
 with open(p,"rb") as f:
  for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
 return h.hexdigest()
def member(root,name):
 if "\\" in name: raise ValueError("Unsafe ZIP path")
 q=PurePosixPath(name)
 if q.is_absolute() or ".." in q.parts: raise ValueError("Unsafe ZIP path")
 p=(Path(root).resolve()/Path(*q.parts)).resolve()
 if p!=Path(root).resolve() and Path(root).resolve() not in p.parents: raise ValueError("ZIP escape")
 return p
def verify_dimer_zip(path):
 path=Path(path); outer=sha256_file(path)
 if EXPECTED_DIMER_ZIP_SHA256 and outer.lower()!=EXPECTED_DIMER_ZIP_SHA256.strip().lower(): raise ValueError("DIMER ZIP SHA-256 mismatch")
 root=Path("/content/dimer-base-model").resolve(); shutil.rmtree(root,ignore_errors=True); root.mkdir()
 total=0
 with zipfile.ZipFile(path) as z:
  for i in z.infolist():
   if stat.S_ISLNK((i.external_attr>>16)&0xffff): raise ValueError("Symlink not allowed")
   total+=i.file_size
   if total>20*1024**3: raise ValueError("DIMER ZIP expands beyond 20 GiB")
   p=member(root,i.filename)
   if i.is_dir(): p.mkdir(parents=True,exist_ok=True); continue
   p.parent.mkdir(parents=True,exist_ok=True)
   with z.open(i) as src,open(p,"wb") as dst: shutil.copyfileobj(src,dst)
 manifests=list(root.rglob("dimer-base-manifest.json"))
 if len(manifests)!=1: raise ValueError("Expected one dimer-base-manifest.json")
 model_root=manifests[0].parent; m=json.loads(manifests[0].read_text())
 if m.get("format")!="dimer_hf_snapshot" or m.get("formatVersion")!=1: raise ValueError("Unsupported DIMER snapshot format")
 if (m.get("modelKey"),m.get("modelId"),m.get("revision"))!=(BASE_MODEL_KEY,model_id,revision): raise ValueError("DIMER model identity mismatch")
 listed=set(); size=0
 for r in m.get("files",[]):
  p=member(model_root,r["path"])
  if not p.is_file() or p.stat().st_size!=r["bytes"] or sha256_file(p)!=r["sha256"]: raise ValueError(f"DIMER file verification failed: {r['path']}")
  listed.add(r["path"]); size+=r["bytes"]
 actual={p.relative_to(model_root).as_posix() for p in model_root.rglob("*") if p.is_file() and p.name!="dimer-base-manifest.json"}
 if actual!=listed or size!=m.get("totalBytes"): raise ValueError("DIMER manifest/file-set mismatch")
 if not any(x.endswith(".safetensors") for x in listed): raise ValueError("No safetensors weights")
 return model_root,outer

MODEL_LOAD_REF=model_id; BASE_MODEL_ACQUISITION={"source":MODEL_SOURCE,"modelId":model_id,"revision":revision}
if MODEL_SOURCE=="DIMER ZIP":
 if UPLOAD_DIMER_ZIP:
  from google.colab import files
  u=files.upload()
  if len(u)!=1: raise ValueError("Upload exactly one DIMER ZIP")
  Path(DIMER_ZIP_PATH).write_bytes(next(iter(u.values())))
 MODEL_LOAD_REF,dsha=verify_dimer_zip(DIMER_ZIP_PATH); BASE_MODEL_ACQUISITION.update({"packageFormat":"dimer_hf_snapshot","zipSha256":dsha})
 print("✓ DIMER base-model package verified")


## 4. Dataset and structural validation

SFT examples describe desired assistant behavior rather than tabular features and a target column. We normalize three common forms into `messages`: chat records; `prompt` plus `completion`; and `instruction` plus optional context and an `output`.

The default Filipino/English seed dataset is useful for teaching the mechanics of SFT but is **not a benchmark**. Dolly is an English fallback. BYOD accepts `train.jsonl`, optional `validation.jsonl`/`val.jsonl`, and optional `test.jsonl`, directly or inside one ZIP.

The notebook rejects unsupported schemas, missing assistant targets, ambiguous validation files, and exact leakage between splits. Exact duplicates inside one split are reported rather than silently deleted. Validation should tell you what is wrong instead of quietly changing your data.


In [ ]:
DATA_SOURCE = "Sample: Filipino SFT" # @param ["Sample: Filipino SFT","Sample: Dolly","Bring Your Own Dataset"]
SAMPLE_LIMIT = 120 # @param {type:"integer"}
MAX_TOTAL_TRAIN_TOKENS = 50_000_000
W=Path("/content/lm-sft"); shutil.rmtree(W,ignore_errors=True); W.mkdir()
def canonical(r):
 if "messages" in r: m=[{"role":x["role"],"content":str(x["content"])} for x in r["messages"]]
 elif "prompt" in r and ("completion" in r or "response" in r): m=[{"role":"user","content":str(r["prompt"])},{"role":"assistant","content":str(r.get("completion",r.get("response")))}]
 elif "instruction" in r and ("output" in r or "response" in r):
  q=str(r["instruction"])+(f"\n\n{r.get('input') or r.get('context')}" if r.get("input") or r.get("context") else ""); m=[{"role":"user","content":q},{"role":"assistant","content":str(r.get("output",r.get("response")))}]
 else: raise ValueError("Unsupported SFT schema")
 if any(x["role"] not in {"system","user","assistant"} for x in m) or not any(x["role"]=="assistant" and x["content"].strip() for x in m): raise ValueError("Invalid roles/assistant target")
 return {"messages":m}
def fp(r): return hashlib.sha256(json.dumps(r,sort_keys=True,ensure_ascii=False,separators=(",",":")).encode()).hexdigest()
if DATA_SOURCE.startswith("Sample"):
 dsid="jpaulpoliquit/ph-sft-ai-authored-v1" if DATA_SOURCE=="Sample: Filipino SFT" else "databricks/databricks-dolly-15k"
 rev=HfApi().dataset_info(dsid).sha if "ph-sft" in dsid else "bdd27f4d94b9c1f951818a7da7fd7aeea5dbff1a"
 lic="apache-2.0" if "ph-sft" in dsid else "cc-by-sa-3.0"
 rows=sorted([canonical(dict(x)) for x in load_dataset(dsid,revision=rev,split="train")],key=fp)[:SAMPLE_LIMIT]; cut=max(1,len(rows)//5)
 SPLITS={"train":rows[cut:],"validation":rows[:cut]}; DATASET_PROVENANCE={"source":dsid,"revision":rev,"license":lic,"usage":"tutorial-training-not-benchmark"}
else:
 from google.colab import files
 u=files.upload(); root=W/"byod"; root.mkdir()
 if len(u)==1 and next(iter(u)).lower().endswith(".zip"):
  zp=W/"data.zip"; zp.write_bytes(next(iter(u.values())))
  with zipfile.ZipFile(zp) as z:
   for i in z.infolist():
    p=member(root,i.filename)
    if stat.S_ISLNK((i.external_attr>>16)&0xffff): raise ValueError("Unsafe dataset ZIP")
    if i.is_dir(): p.mkdir(parents=True,exist_ok=True); continue
    p.parent.mkdir(parents=True,exist_ok=True)
    with z.open(i) as src,open(p,"wb") as dst: shutil.copyfileobj(src,dst)
 else:
  for n,b in u.items(): (root/Path(n).name).write_bytes(b)
 def read(p): return [canonical(json.loads(x)) for x in p.read_text().splitlines() if x.strip()]
 if not (root/"train.jsonl").exists(): raise ValueError("BYOD requires train.jsonl")
 if (root/"validation.jsonl").exists() and (root/"val.jsonl").exists(): raise ValueError("Ambiguous validation split")
 SPLITS={"train":read(root/"train.jsonl")}; v=root/("validation.jsonl" if (root/"validation.jsonl").exists() else "val.jsonl")
 if v.exists(): SPLITS["validation"]=read(v)
 if (root/"test.jsonl").exists(): SPLITS["test"]=read(root/"test.jsonl")
 DATASET_PROVENANCE={"source":"BYOD","usage":"user-provided"}
for k,v in SPLITS.items():
 if len(v)!=len({fp(x) for x in v}): print(f"Warning: exact duplicates in {k}; none removed")
for a,b in [("train","validation"),("train","test"),("validation","test")]:
 if a in SPLITS and b in SPLITS and {fp(x) for x in SPLITS[a]}&{fp(x) for x in SPLITS[b]}: raise ValueError(f"Split leakage {a}/{b}")
if "validation" not in SPLITS: rows=SPLITS["train"]; cut=max(1,len(rows)//5); SPLITS={**SPLITS,"train":rows[cut:],"validation":rows[:cut]}
DATASET_DIGEST=hashlib.sha256("".join(fp(x) for k in sorted(SPLITS) for x in SPLITS[k]).encode()).hexdigest(); print({k:len(v) for k,v in SPLITS.items()},DATASET_DIGEST[:16])


## 5. Tokenizer validation and assistant-only masking

The tokenizer is part of the model interface: it converts text into token IDs and its chat template decides how `system`, `user`, and `assistant` roles are serialized. Tokenizer-dependent checks therefore happen only after a base model is selected.

For supervised fine-tuning, we usually want the loss to supervise the **assistant response**, not teach the model to reproduce the user's question. We set non-assistant labels to `-100`, PyTorch's ignore index, and leave only assistant spans trainable. Those spans are computed by incrementally rendering the conversation with the model's own chat template.

The notebook checks token-prefix stability and refuses over-length records rather than silently truncating them. Silent truncation can alter the intended target and makes a run harder to audit.


In [ ]:
tok_kw={"trust_remote_code":False}
if MODEL_SOURCE=="Pinned Hugging Face": tok_kw.update({"revision":revision,**({"token":HF_TOKEN} if HF_TOKEN else {})})
else: tok_kw["local_files_only"]=True
tokenizer=AutoTokenizer.from_pretrained(MODEL_LOAD_REF,**tok_kw)
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
if not tokenizer.chat_template: raise ValueError("No chat template")
IGNORE=-100
def enc(s): return tokenizer(s,add_special_tokens=False)["input_ids"]
def rend(m,g=False): return tokenizer.apply_chat_template(m,tokenize=False,add_generation_prompt=g)
def build_masked_example(r):
 m=r["messages"]; ids=enc(rend(m)); labels=[IGNORE]*len(ids)
 if len(ids)>MAX_SEQUENCE_LENGTH: raise ValueError("DATASET_SEQUENCE_TOO_LONG")
 for i,x in enumerate(m):
  if x["role"]=="assistant":
   a,b=enc(rend(m[:i],True)),enc(rend(m[:i+1]))
   if ids[:len(a)]!=a or ids[:len(b)]!=b: raise ValueError("Chat template not prefix-stable")
   labels[len(a):len(b)]=ids[len(a):len(b)]
 if all(x==IGNORE for x in labels): raise ValueError("No supervised tokens")
 return ids,labels
MASKED={k:[build_masked_example(r) for r in v] for k,v in SPLITS.items()}; totals={k:sum(len(x[0]) for x in v) for k,v in MASKED.items()}
if totals["train"]>MAX_TOTAL_TRAIN_TOKENS: raise ValueError("DATASET_TOKEN_BUDGET_EXCEEDED")
print(totals)


## 6. Baseline generation, QLoRA, and optimization evidence

Before training, we save responses to fixed prompts. Reusing exactly the same prompts afterward makes behavior changes visible and avoids attributing ordinary base-model behavior to the adapter.

QLoRA keeps the frozen base weights quantized to 4-bit NF4 and trains small LoRA matrices attached to projection layers. The exported result is therefore an **adapter**, not another full copy of the base model.

The training loop reports train/validation loss, validation perplexity where finite, runtime, and peak allocated GPU memory. These are **optimization** measurements. Lower loss may show that the model fit the supervised token objective better, but it does not prove factuality, safety, cultural appropriateness, or downstream usefulness. A real application still needs an independent task-specific evaluation.


In [ ]:
bf16=torch.cuda.is_bf16_supported(); dtype=torch.bfloat16 if bf16 else torch.float16
q=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=dtype)
load_kw={"trust_remote_code":False,"dtype":dtype,"quantization_config":q,"device_map":{"":0}}
if MODEL_SOURCE=="Pinned Hugging Face": load_kw.update({"revision":revision,**({"token":HF_TOKEN} if HF_TOKEN else {})})
else: load_kw["local_files_only"]=True
base_model=AutoModelForCausalLM.from_pretrained(MODEL_LOAD_REF,**load_kw)
def generate(m,p,n=96):
 s=rend([{"role":"user","content":p}],True); x=tokenizer(s,return_tensors="pt",add_special_tokens=False).to("cuda")
 with torch.no_grad(): y=m.generate(**x,max_new_tokens=n,do_sample=False,pad_token_id=tokenizer.pad_token_id)
 return tokenizer.decode(y[0,x["input_ids"].shape[1]:],skip_special_tokens=True).strip()
PROMPTS=["Ipaliwanag sa simpleng Filipino kung ano ang machine learning.","Magbigay ng tatlong paraan para mabawasan ang basura sa opisina."]
BASELINE_OUTPUTS=[generate(base_model,p) for p in PROMPTS]
targets=sorted({n.rsplit(".",1)[-1] for n,_ in base_model.named_modules()}&{"q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"})
if not targets: raise RuntimeError("No LoRA target modules found")
model=get_peft_model(prepare_model_for_kbit_training(base_model),LoraConfig(r=LORA_RANK,lora_alpha=16,lora_dropout=.05,bias="none",task_type="CAUSAL_LM",target_modules=targets))
def batch(x):
 ids,lab=x; return {"input_ids":torch.tensor([ids],device="cuda"),"labels":torch.tensor([lab],device="cuda"),"attention_mask":torch.ones((1,len(ids)),dtype=torch.long,device="cuda")}
def evaluate(xs):
 if not xs: return None
 model.eval(); loss=toks=0
 with torch.no_grad():
  for x in xs:
   b=batch(x); o=model(**b); n=int((b["labels"]!=IGNORE).sum()); loss+=float(o.loss)*n; toks+=n
 model.train(); return loss/toks if toks else None
opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=LEARNING_RATE); started=time.time(); torch.cuda.reset_peak_memory_stats(); random.seed(SEED)
for e in range(EPOCHS):
 xs=MASKED["train"][:]; random.shuffle(xs); loss=toks=0; opt.zero_grad()
 for i,x in enumerate(xs,1):
  b=batch(x); o=model(**b); (o.loss/2).backward(); n=int((b["labels"]!=IGNORE).sum()); loss+=float(o.loss)*n; toks+=n
  if i%2==0 or i==len(xs): opt.step(); opt.zero_grad()
 val=evaluate(MASKED["validation"]); print(e+1,loss/toks,val)
METRICS={"trainLoss":loss/toks,"validationLoss":val,"testLoss":evaluate(MASKED.get("test",[])),"validationPerplexity":math.exp(val) if val is not None and val<20 else None,"wallSeconds":time.time()-started,"peakGpuMemoryBytes":torch.cuda.max_memory_allocated(),"peakGpuMemoryGiB":torch.cuda.max_memory_allocated()/1024**3}
ADAPTED_OUTPUTS=[generate(model,p) for p in PROMPTS]
display(pd.DataFrame({"prompt":PROMPTS,"base":BASELINE_OUTPUTS,"adapted":ADAPTED_OUTPUTS})); print(METRICS)


## 7. New-prompt inference

The before/after table is useful for teaching, but it intentionally reuses prompts. This section asks the adapted model something new, analogous to “new-data inference” in the tabular tutorials.

Edit the prompts to probe the behavior that matters to you. Interactive exploration helps reveal obvious regressions or language/style changes, but it is not a statistically meaningful evaluation. Define a separate held-out evaluation protocol before using the model operationally.


In [ ]:
RUN_NEW_PROMPT_INFERENCE = True # @param {type:"boolean"}
NEW_PROMPTS=["Sumulat ng maikling payo para sa isang estudyanteng nagsisimula sa AI.","Ipaliwanag ang pagkakaiba ng training data at evaluation data sa dalawang pangungusap."]
if RUN_NEW_PROMPT_INFERENCE: display(pd.DataFrame({"prompt":NEW_PROMPTS,"response":[generate(model,p) for p in NEW_PROMPTS]}))


## 8. Adapter-first export and clean reload

The portable result of QLoRA is the PEFT adapter plus provenance, not another copy of the full base weights. We save the adapter with safetensors, the tokenizer, metrics, provenance, a model card, and an `artifact-manifest.json` that hashes every published file.

The provenance records the exact base model/revision, whether credentialed access was required, the model-acquisition route, dataset identity/digest, training settings, and runtime context. It **does not store `HF_TOKEN`** and it does not serialize training rows.

Before publication we delete the in-memory training model, reacquire the base from the same source, attach the adapter **from disk**, and generate a smoke response. That is stronger evidence than generating from the object that just finished training: it demonstrates that the exported files are sufficient to reconstruct the adapted model.


In [ ]:
S=Path("/content/dimer-lm-adapter.staging"); A=Path("/content/dimer-lm-adapter"); Z=Path("/content/dimer-language-model-adapter.zip")
shutil.rmtree(S,ignore_errors=True); shutil.rmtree(A,ignore_errors=True); S.mkdir()
model.save_pretrained(S,safe_serialization=True); tokenizer.save_pretrained(S/"tokenizer")
PROVENANCE={"artifactFormat":"peft_adapter","artifactFormatVersion":1,"baseModel":model_id,"baseModelRevision":revision,"baseModelRevisionExpected":revision,"baseModelLicense":base_license,"modelKey":BASE_MODEL_KEY,"trustRemoteCode":False,"requiresHfToken":bool(entry["requires_hf_token"]),"dimerZipAllowed":bool(entry["dimer_zip"]),"baseModelAcquisition":BASE_MODEL_ACQUISITION,"datasetDigest":DATASET_DIGEST,"dataset":DATASET_PROVENANCE,"training":{"method":TRAINING_METHOD,"epochs":EPOCHS,"learningRate":LEARNING_RATE,"loraRank":LORA_RANK},"runtime":{"python":platform.python_version(),"torch":torch.__version__,"gpu":GPU_NAME,"gpuVramGiB":GPU_VRAM_GB},"aiProvenance":AI_PROVENANCE}
(S/"metrics.json").write_text(json.dumps(METRICS,indent=2)); (S/"provenance.json").write_text(json.dumps(PROVENANCE,indent=2)); (S/"MODEL_CARD.md").write_text(f"# PEFT adapter for {model_id}\n\nBase revision: `{revision}`. Optimization metrics are not task-quality evidence.\n")
records=[{"path":p.relative_to(S).as_posix(),"bytes":p.stat().st_size,"sha256":sha256_file(p)} for p in sorted(S.rglob("*")) if p.is_file() and p.name!="artifact-manifest.json"]
(S/"artifact-manifest.json").write_text(json.dumps({"format":"peft_adapter","formatVersion":1,"files":records,"totalBytes":sum(x["bytes"] for x in records)},indent=2))
for r in records:
 if sha256_file(S/r["path"])!=r["sha256"]: raise RuntimeError("artifact-manifest.json verification failed")
# Fresh base + adapter reload, from disk, before publication.
del model,base_model; gc.collect(); torch.cuda.empty_cache()
reload_kw={"trust_remote_code":False,"dtype":dtype,"quantization_config":q,"device_map":{"":0}}
if MODEL_SOURCE=="Pinned Hugging Face": reload_kw.update({"revision":revision,**({"token":HF_TOKEN} if HF_TOKEN else {})})
else: reload_kw["local_files_only"]=True
rb=AutoModelForCausalLM.from_pretrained(MODEL_LOAD_REF,**reload_kw); rt=AutoTokenizer.from_pretrained(S/"tokenizer",local_files_only=True,trust_remote_code=False)
rm=PeftModel.from_pretrained(rb,S,is_trainable=False); tokenizer=rt; smoke=generate(rm,"Kumusta! Sagutin sa isang maikling pangungusap.",32)
if not smoke: raise RuntimeError("Fresh reload failed")
print("✓ Fresh base + adapter reload",smoke); os.replace(S,A)
with zipfile.ZipFile(Z,"w",zipfile.ZIP_STORED) as z:
 for p in A.rglob("*"):
  if p.is_file(): z.write(p,p.relative_to(A).as_posix())
print("Artifact SHA-256",sha256_file(Z))
from google.colab import files
files.download(str(Z))


## 9. What a successful run proves

If the notebook completes, the **exact path you selected** has executor evidence: pinned model acquisition, structural and tokenizer-aware validation, assistant-only QLoRA SFT, baseline/adapted generation, fresh-prompt inference, hashed adapter export, and fresh base-plus-adapter reload.

For SmolLM3, the printed runtime and peak VRAM are measurements for this particular Colab run, not automatically a universal production minimum. For Llama, successful secret preflight and loading show that `HF_TOKEN` in Colab can clear the credential gate for an account that accepted the terms. DIMER credential delivery remains a separate platform test, and DIMER-hosted redistribution remains a separate licensing decision.

A successful optimization run does **not** prove universal model quality, factuality, safety, fairness, or deployment fitness. That requires an independent evaluation designed for the intended application.
